# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sanaanwar25/flyrank-ml-internship-sana/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build the feature vector from information available in the February feature window. Numeric features are aggregated at the client × content level, missing numeric values are filled with 0, and categorical fields are filled with an explicit "missing" value. March outcome fields and label-derived fields are excluded.

In [11]:
import os
import pandas as pd
import numpy as np

# Make sure the starter repo is available
repo = "/content/flyrank-ml-internship-starter"

if not os.path.exists(repo):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

os.chdir(repo)

# Load the starter data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

Rows: 30000
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

impressions_feb and clicks_feb measure February search exposure and clicks. ctr_feb measures February click-through rate. avg_position_feb measures average February search position. impressions_feb_total and clicks_feb_total summarize February activity. content_age_days represents content age and search_volume represents observed search demand. Numeric missing values are filled with 0. These features are available before the March prediction window.

In [12]:
# Check feature columns and missing values

print("Feature dataframe shape:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

print("\nData types:")
print(df.dtypes)

Feature dataframe shape: (30000, 44)

Missing values:
content_id                    0
client_id                     0
search_volume              2468
competition                2468
competition_level          2610
cpc                        2468
content_type                  0
main_intent                2374
word_count                 7699
char_count                 7699
provider_used             21438
model_used                 5733
impressions_90d               0
clicks_90d                    0
pageviews_90d                 0
sessions_90d                  0
users_90d                     0
engaged_sessions_90d          0
ai_sessions_90d               0
scroll_events_90d             0
days_with_impressions         0
days_with_sessions            0
impressions_last_30d          0
clicks_last_30d               0
sessions_last_30d             0
impressions_prev_30d          0
clicks_prev_30d               0
sessions_prev_30d             0
content_age_days              0
age_tier          

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked for fields that could contain the future outcome or directly reproduce the label. Future-window outcome fields are excluded from the feature set. I also check for suspicious names such as impressions, sessions, clicks, or outcome fields before the prediction window.

In [13]:
# Leakage check: inspect column names for likely future/outcome fields

suspicious_words = [
    "label", "target", "outcome",
    "impressions_90d", "sessions_90d",
    "clicks_90d", "future"
]

suspicious = [
    c for c in df.columns
    if any(word in c.lower() for word in suspicious_words)
]

print("Potential leakage/outcome columns:")
print(suspicious)

Potential leakage/outcome columns:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d', 'ai_sessions_90d']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I exclude future outcome fields because they are only known after the prediction point. I exclude direct label or target fields because they would leak the answer into the features. I also exclude identifiers that do not represent useful predictive information.

In [14]:
# Show columns that look like identifiers or future outcomes

excluded_candidates = [
    c for c in df.columns
    if any(word in c.lower() for word in [
        "label", "target", "outcome",
        "impressions_90d", "sessions_90d",
        "client_hash_id", "content_hash_id"
    ])
]

print("Excluded candidates:")
print(excluded_candidates)

Excluded candidates:
['impressions_90d', 'sessions_90d', 'engaged_sessions_90d', 'ai_sessions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.